# SparkClient: Handling Batch Job Failures and Log Retrieval

This notebook demonstrates how to submit a Spark job expected to fail, wait for a failure status, retrieve crash logs for diagnosis, and clean up resources.

### What you will learn:
1. Submitting a remote `FileJob` whose application intentionally raises an exception.
2. Waiting for a job to reach `SparkJobStatus.FAILED`.
3. Inspecting the failed job status and driver pod metadata.
4. Streaming driver logs to diagnose the failure reason.
5. Deleting the failed Spark job.

## 1. Imports and Client Initialization

Import necessary classes and initialize `SparkClient` with the target namespace.

In [ ]:
import os

from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.spark import FileJob, SparkClient, SparkJobStatus

namespace = os.environ.get("SPARK_TEST_NAMESPACE", "default")
backend_config = KubernetesBackendConfig(namespace=namespace)
client = SparkClient(backend_config=backend_config)

print(f"SparkClient initialized for namespace: {namespace}")

## 2. Submit Job Expected to Fail

Submit the standalone `spark_failed_job.py` application from this repository. It runs a small DataFrame computation and then raises a `RuntimeError` on purpose, so the `SparkApplication` ends in `FAILED`.

In [ ]:
REMOTE_JOB = (
    "https://raw.githubusercontent.com/kubeflow/sdk/main/examples/spark/spark_failed_job.py"
)

print("Submitting Spark job expected to fail...")
job_name = client.submit_job(
    job=FileJob(
        file_source=REMOTE_JOB,
    ),
    num_executors=1,
    resources_per_executor={
        "cpu": "1",
        "memory": "512Mi",
    },
)

print(f"Job submitted successfully: {job_name}")

## 3. Wait for Failure Status

Monitor the application and wait until it transitions to `SparkJobStatus.FAILED`.

In [ ]:
print(f"Waiting for {job_name} to fail...")
job = client.wait_for_job_status(
    job_name,
    status={SparkJobStatus.FAILED},
    timeout=300,
)
assert job is not None, f"Expected job {job_name} to exist"
assert job.status == SparkJobStatus.FAILED, f"Expected FAILED status, got {job.status}"

print("Job failed as expected.")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Namespace: {job.namespace}")

## 4. Get Job Details

Retrieve job metadata using `get_job` to confirm application failure state.

In [ ]:
job = client.get_job(job_name)
assert job is not None, f"Failed to retrieve metadata for {job_name}"
assert job.status == SparkJobStatus.FAILED, f"Expected FAILED status in get_job, got {job.status}"

print("Job details:")
print(f"Name: {job.name}")
print(f"Namespace: {job.namespace}")
print(f"Status: {job.status}")
print(f"Driver Pod: {job.driver_pod_name}")
print(f"Executors: {job.num_executors}")

## 5. Retrieve Driver Crash Logs

Read the driver logs, show the traceback, and confirm that the job failed with the intentional `RuntimeError` raised by `spark_failed_job.py` rather than for another reason, such as the application file not being found. The traceback is near the end of the logs, after Spark's startup and shutdown messages.

In [ ]:
print(f"Retrieving logs for: {job_name}")
print("-" * 70)

logs = list(client.get_job_logs(job_name))
line_count = 0
for line in logs:
    print(line.rstrip())
    line_count += 1
    if line_count >= 20:
        print("...")
        break

print("-" * 70)
print(f"Displayed {line_count} log lines.")
assert line_count > 0, f"Expected driver crash logs to be present for {job_name}"
assert any("Intentional failure for Spark SDK E2E testing" in line for line in logs), (
    f"Expected {job_name} to fail with the intentional error"
)

## 6. Clean Up Resources

Delete the failed Spark job from the cluster.

In [ ]:
print(f"Deleting job: {job_name}")
client.delete_job(job_name)
print("Job deleted successfully.")